In [0]:
%pip install simpledbf geopandas --quiet

%restart_python

## Cells 1 - Imports & Logging

In [0]:
import re
import os
import logging
import zipfile
import tempfile
import requests
import pandas as pd

from datetime import datetime, timezone
from io import BytesIO

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s"
)
logger = logging.getLogger("stl_parcel_ingest")

print("✓ Imports loaded")

## Cells 2 - Configuration

In [0]:
# Defining the output path
RAW_OUTPUT_PATH = "/Volumes/workspace/default/raw/parcels"

print(f"✓ Output path: {RAW_OUTPUT_PATH}")

## Cell 3 — Source URLs

In [0]:
# All URLs confirmed from dynamic.stlouis-mo.gov/opendata/downloads.cfm
# These are static files — the city overwrites them in place on each update.
# No date logic needed, just download and parse.

# We only need two sources — the DBF contains address, neighborhood,
# ward, assessment values, AND vacancy status, so the separate
# vacancy survey download is unnecessary.

PARCEL_SHAPEFILE_URL = (
    "https://dynamic.stlouis-mo.gov/opendata/downloads/prcl_shape.zip"
)

LAND_RECORDS_URL = (
    "https://dynamic.stlouis-mo.gov/opendata/downloads/par.zip"
)

print("✓ Source URLs set")
print(f"  Shapefile:    {PARCEL_SHAPEFILE_URL}")
print(f"  Land Records: {LAND_RECORDS_URL}")

## Cell 4 — Canonical Schema

In [0]:
# The output schema for the enriched parcel DataFrame.
# Kept deliberately narrow — only fields needed for the
# neighborhood intelligence model downstream.
#
# All three source files join on parcel_id (the HANDLE field).
# Fields with None as default will be null for parcels where
# the source file didn't have a record (e.g. vacant land parcels
# may not appear in the assessment data).

CANONICAL_SCHEMA: dict[str, object] = {
    "parcel_id":        StringType(),   # HANDLE — join key across sources
    "address":          StringType(),   # SITEADDR from DBF
    "neighborhood":     StringType(),   # NBRHD from DBF (numeric code)
    "ward":             StringType(),   # WARD from DBF
    "latitude":         DoubleType(),   # centroid from shapefile (WGS84)
    "longitude":        DoubleType(),   # centroid from shapefile (WGS84)
    "appraised_value":  DoubleType(),   # ASMTLAND from DBF
    "assessed_value":   DoubleType(),   # ASMTTOTAL from DBF
    "is_vacant":        StringType(),   # VACANTLAND from DBF — "Y"/"N"
}

print(f"✓ Canonical schema: {len(CANONICAL_SCHEMA)} fields")
for field, dtype in CANONICAL_SCHEMA.items():
    print(f"  {field:20s} {dtype}")

## Cell 5 — HTTP Fetch Helper

In [0]:
BROWSER_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

def fetch_zip(url: str, timeout: int = 120) -> bytes | None:
    """
    Download a ZIP file from the given URL.
    Returns raw bytes on success, None on failure.

    Timeout is higher than the crime notebook (120s vs 60s) because
    the parcel files are larger — the Tax Records ZIP is 69MB and
    even the shapefile is 7MB.
    """
    for attempt in range(1, 4):
        try:
            logger.info(f"  GET {url} (attempt {attempt}/3)")
            resp = requests.get(url, headers=BROWSER_HEADERS, timeout=timeout)

            if resp.status_code == 200:
                logger.info(f"  ✓ {len(resp.content):,} bytes received")
                return resp.content

            if resp.status_code == 403:
                logger.warning(f"  403 — forbidden, not retrying")
                return None

            if resp.status_code == 404:
                logger.warning(f"  404 — not found: {url}")
                return None

            logger.warning(f"  HTTP {resp.status_code} on attempt {attempt}/3")

        except requests.Timeout:
            logger.warning(f"  Timeout on attempt {attempt}/3")
        except requests.RequestException as e:
            logger.warning(f"  Request error: {e}")

    logger.error(f"  All attempts failed for {url}")
    return None


print("✓ fetch_zip() defined")

## Cell 6 — Parse Shapefile

In [0]:
def parse_shapefile(zip_bytes: bytes) -> pd.DataFrame | None:
    """
    Extract parcel_id and centroid coordinates from the city
    parcel shapefile ZIP.

    The shapefile only contains HANDLE + geometry — all other
    attributes come from the Land Records DBF. We compute the
    centroid of each parcel polygon to get a representative
    lat/lon point, then return a two-column DataFrame that will
    be joined to the DBF data on parcel_id.

    Important: centroid must be computed in the original projected
    CRS (NAD27 Missouri East, in feet) BEFORE reprojecting to
    WGS84. Computing centroid after reprojection triggers a
    geopandas warning about geographic CRS inaccuracy.

    Args:
        zip_bytes: Raw bytes of the downloaded shapefile ZIP.

    Returns:
        Pandas DataFrame with parcel_id, latitude, longitude
        or None on failure.
    """
    try:
        import geopandas as gpd
    except ImportError:
        logger.error("geopandas not available — run Cell 1 pip install")
        return None

    # ── Step 1: Extract ZIP to temp directory ─────────────────
    with tempfile.TemporaryDirectory() as tmpdir:
        try:
            with zipfile.ZipFile(BytesIO(zip_bytes)) as zf:
                zf.extractall(tmpdir)
        except zipfile.BadZipFile as e:
            logger.error(f"  Bad ZIP: {e}")
            return None

        shp_files = [
            os.path.join(tmpdir, f)
            for f in os.listdir(tmpdir)
            if f.endswith(".shp")
        ]
        if not shp_files:
            logger.error("  No .shp file found in ZIP")
            return None

        logger.info(f"  Reading: {os.path.basename(shp_files[0])}")

        # ── Step 2: Read shapefile ─────────────────────────────
        try:
            gdf = gpd.read_file(shp_files[0])
        except Exception as e:
            logger.error(f"  Read error: {e}")
            return None

        logger.info(f"  Loaded {len(gdf):,} parcels, CRS: {gdf.crs.to_epsg()}")

        # ── Step 3: Compute centroid in projected CRS ──────────
        # Must happen BEFORE reprojection to WGS84. Computing
        # centroids in a projected CRS (units = feet) is accurate.
        # Computing them after reprojection (units = degrees) is
        # not and triggers a geopandas warning.
        gdf["centroid"] = gdf.geometry.centroid

        # ── Step 4: Reproject centroids to WGS84 ──────────────
        # Convert just the centroid points — we don't need the
        # full polygon geometry after this point.
        centroids_wgs84 = gdf["centroid"].set_crs(
            gdf.crs
        ).to_crs(epsg=4326)

        # ── Step 5: Extract lat/lon from reprojected centroids ─
        gdf["latitude"]  = centroids_wgs84.y
        gdf["longitude"] = centroids_wgs84.x

        # ── Step 6: Return just parcel_id + coordinates ────────
        pdf = pd.DataFrame({
            "parcel_id": gdf["HANDLE"].astype(str).str.strip(),
            "latitude":  gdf["latitude"],
            "longitude": gdf["longitude"],
        })

        # Drop any rows where coordinates are null or zero
        pdf = pdf[
            pdf["latitude"].notna()  &
            pdf["longitude"].notna() &
            (pdf["latitude"]  != 0)  &
            (pdf["longitude"] != 0)
        ]

        logger.info(f"  ✓ {len(pdf):,} parcels with valid coordinates")
        return pdf


print("✓ parse_shapefile() defined")

## Cell 7 — Parse the Land Records

In [0]:
def parse_land_records(zip_bytes: bytes) -> pd.DataFrame | None:
    """
    Extract address, neighborhood, ward, assessment values, and
    vacancy status from the Land Records DBF file.

    The DBF uses latin-1 encoding (Windows legacy format) — this
    must be specified at read time or simpledbf will throw a
    UnicodeDecodeError on special characters in owner names and
    addresses.

    We only keep the 7 fields we need for the canonical schema.
    The DBF has 86 columns total — pulling everything would waste
    memory and make the join output unnecessarily wide.

    Args:
        zip_bytes: Raw bytes of the downloaded land records ZIP.

    Returns:
        Pandas DataFrame with parcel_id, address, neighborhood,
        ward, appraised_value, assessed_value, is_vacant
        or None on failure.
    """
    from simpledbf import Dbf5

    # ── Step 1: Extract ZIP ───────────────────────────────────
    with tempfile.TemporaryDirectory() as tmpdir:
        try:
            with zipfile.ZipFile(BytesIO(zip_bytes)) as zf:
                zf.extractall(tmpdir)
        except zipfile.BadZipFile as e:
            logger.error(f"  Bad ZIP: {e}")
            return None

        dbf_files = [
            os.path.join(tmpdir, f)
            for f in os.listdir(tmpdir)
            if f.lower().endswith(".dbf")
        ]
        if not dbf_files:
            logger.error("  No DBF file found")
            return None

        logger.info(f"  Reading: {os.path.basename(dbf_files[0])}")

        # ── Step 2: Read DBF with latin-1 encoding ────────────
        # latin-1 (iso-8859-1) handles the special characters that
        # appear in owner names and addresses in this dataset.
        try:
            dbf = Dbf5(dbf_files[0], codec="latin-1")
            pdf = dbf.to_dataframe()
        except Exception as e:
            logger.error(f"  DBF read error: {e}")
            return None

        logger.info(f"  Loaded {len(pdf):,} rows, {len(pdf.columns)} columns")

        # ── Step 3: Extract only the fields we need ───────────
        # HANDLE     → parcel_id  (join key)
        # SITEADDR   → address    (street address)
        # NBRHD      → neighborhood (numeric neighborhood code)
        # WARD       → ward
        # ASMTLAND   → appraised_value (land assessment)
        # ASMTTOTAL  → assessed_value  (total assessment)
        # VACANTLAND → is_vacant  ("Y"/"N")
        keep = {
            "HANDLE":     "parcel_id",
            "SITEADDR":   "address",
            "NBRHD":      "neighborhood",
            "WARD":       "ward",
            "ASMTLAND":   "appraised_value",
            "ASMTTOTAL":  "assessed_value",
            "VACANTLAND": "is_vacant",
        }

        # Verify all expected columns are present
        missing = [c for c in keep if c not in pdf.columns]
        if missing:
            logger.error(f"  Missing expected columns: {missing}")
            return None

        pdf = pdf[list(keep.keys())].rename(columns=keep)

        # ── Step 4: Clean up types ────────────────────────────
        # parcel_id: strip whitespace from string
        # neighborhood/ward: convert to string (stored as int in DBF)
        # assessment values: already numeric, coerce to float
        # is_vacant: strip whitespace
        pdf["parcel_id"]    = pdf["parcel_id"].astype(str).str.strip()
        pdf["neighborhood"] = pdf["neighborhood"].astype(str).str.strip()
        pdf["ward"]         = pdf["ward"].astype(str).str.strip()
        pdf["is_vacant"]    = pdf["is_vacant"].astype(str).str.strip()

        # Replace "nan" strings produced by .astype(str) on null values
        pdf["is_vacant"] = pdf["is_vacant"].replace("nan", None)

        pdf["appraised_value"] = pd.to_numeric(
            pdf["appraised_value"], errors="coerce"
        )
        pdf["assessed_value"] = pd.to_numeric(
            pdf["assessed_value"], errors="coerce"
        )

        logger.info(f"  ✓ {len(pdf):,} land records extracted")
        return pdf


print("✓ parse_land_records() defined")

## Cell 8 — Join & Build Final DataFrame

In [0]:
def build_parcel_df(spark: SparkSession) -> DataFrame | None:
    """
    Download both sources, parse them, join on parcel_id, and
    return a normalized Spark DataFrame matching CANONICAL_SCHEMA.

    Join strategy:
      - Base: Land Records DBF (127,001 rows — has all attributes)
      - Left join: Shapefile centroids (126,962 rows — has lat/lon)
      - A small number of parcels may not have coordinates if they
        are in the DBF but not the shapefile — these get null lat/lon.

    Args:
        spark: Active SparkSession.

    Returns:
        Normalized Spark DataFrame or None if either source fails.
    """

    # ── Download and parse Land Records (base table) ──────────
    logger.info("Fetching land records DBF...")
    raw_dbf = fetch_zip(LAND_RECORDS_URL)
    if raw_dbf is None:
        logger.error("Land records download failed")
        return None

    dbf_pdf = parse_land_records(raw_dbf)
    if dbf_pdf is None:
        return None

    # ── Download and parse Shapefile (coordinates only) ───────
    logger.info("Fetching parcel shapefile...")
    raw_shp = fetch_zip(PARCEL_SHAPEFILE_URL)
    if raw_shp is None:
        logger.error("Shapefile download failed")
        return None

    shp_pdf = parse_shapefile(raw_shp)
    if shp_pdf is None:
        return None

    # ── Join on parcel_id ─────────────────────────────────────
    # Left join keeps all DBF records even if the shapefile
    # doesn't have a matching geometry for that parcel.
    logger.info("Joining land records to shapefile coordinates...")
    merged = dbf_pdf.merge(shp_pdf, on="parcel_id", how="left")
    logger.info(f"  Merged: {len(merged):,} rows")

    # ── Convert to Spark and cast to canonical types ──────────
    sdf = spark.createDataFrame(merged)

    for col_name, spark_type in CANONICAL_SCHEMA.items():
        if col_name in sdf.columns:
            sdf = sdf.withColumn(col_name, F.col(col_name).cast(spark_type))

    # ── Attach provenance metadata ────────────────────────────
    sdf = sdf.withColumn("_source", F.lit("stlouis_parcels"))
    sdf = sdf.withColumn(
        "_ingested_at",
        F.lit(datetime.now(timezone.utc).isoformat())
    )

    return sdf


print("✓ build_parcel_df() defined")

## Cell 9 — Run Ingestion & Write Parquet

In [0]:
print("Starting parcel ingestion...")
print("=" * 60)

parcel_df = build_parcel_df(spark)

if parcel_df is None:
    raise RuntimeError("Parcel ingestion failed — check logs above")

# Write to Parquet — parcels are a single snapshot (no partitioning
# needed since this isn't time-series data like crime)
parcel_df.write.mode("overwrite").parquet(RAW_OUTPUT_PATH)

row_count = parcel_df.count()
print(f"\n✓ Wrote {row_count:,} rows → {RAW_OUTPUT_PATH}")
parcel_df.printSchema()

## Cell 10 — Quality Checks

In [0]:
print("=== NULL RATES BY COLUMN (%) ===")
display(parcel_df.select([
    F.round(
        F.count(F.when(F.col(c).isNull(), c)) / F.count(F.lit(1)) * 100, 2
    ).alias(c)
    for c in CANONICAL_SCHEMA.keys()
]))

# COMMAND ----------

print("=== VACANCY DISTRIBUTION ===")
display(
    parcel_df
    .groupBy("is_vacant")
    .count()
    .orderBy("count", ascending=False)
)

# COMMAND ----------

print("=== ASSESSMENT VALUE DISTRIBUTION ===")
display(
    parcel_df.select(
        F.min("assessed_value").alias("min"),
        F.max("assessed_value").alias("max"),
        F.round(F.avg("assessed_value"), 2).alias("avg"),
        F.percentile_approx("assessed_value", 0.5).alias("median"),
    )
)

# COMMAND ----------

print("=== TOP 20 NEIGHBORHOODS BY PARCEL COUNT ===")
display(
    parcel_df
    .groupBy("neighborhood")
    .count()
    .orderBy("count", ascending=False)
    .limit(20)
)

## Cell 11 — Verify Parquet Write

In [0]:
persisted_df    = spark.read.parquet(RAW_OUTPUT_PATH)
persisted_count = persisted_df.count()

print(f"Rows read back from Parquet: {persisted_count:,}")

if persisted_count == row_count:
    print("✓ Row count matches — write verified")
else:
    print(f"⚠ Mismatch: in-memory={row_count:,}, on-disk={persisted_count:,}")